# Deep Learning - Topic 6: Energy-Based Models (EBM)

## EBM on Two Moons Dataset

This notebook uses `sklearn.datasets.make_moons` to create a two-moons dataset, trains an EBM to learn its distribution, visualizes the energy and generated samples, and evaluates the model with KL divergence and Jensen-Shannon distance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras import layers
import sklearn.datasets
import scipy.stats
import scipy.spatial.distance
from functools import partial

## 1. Create and visualize the two-moons dataset

Generate the dataset with `make_moons` and plot the points. Then estimate and visualize the density using a Gaussian KDE.

In [ ]:
# Create two-moons dataset
n_samples = 2000
noise = 0.1
X_moons, y_moons = sklearn.datasets.make_moons(n_samples=n_samples, noise=noise, random_state=42)
print("Dataset shape:", X_moons.shape)
plt.figure(figsize=(6, 5))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap="viridis", alpha=0.6, s=10)
plt.title("Two Moons Dataset")
plt.xlabel("x1")
plt.ylabel("x2")
plt.colorbar(label="Class")
plt.tight_layout()
plt.show()

In [ ]:
# Density estimate (KDE) for the dataset
def create_contour_plot(f, transpose=False, x_lim=(-2, 3), y_lim=(-1.5, 2), num_points=250, show_plot=True):
    x_surface = np.linspace(x_lim[0], x_lim[1], num_points)
    y_surface = np.linspace(y_lim[0], y_lim[1], num_points)
    X, Y = np.meshgrid(x_surface, y_surface)
    combined = np.vstack((X.flatten(), Y.flatten())).T
    if transpose:
        Z = f(combined.T)
    else:
        Z = f(combined)
    plt.contourf(X, Y, Z.reshape(X.shape), levels=20)
    plt.colorbar()
    if show_plot:
        plt.show()
    return Z

kernel = scipy.stats.gaussian_kde(X_moons.T, bw_method=0.15)
plt.figure(figsize=(6, 5))
plt.title("Density estimate (KDE) of the two-moons dataset")
plt.xlabel("x1")
plt.ylabel("x2")
Z_kde = create_contour_plot(kernel, transpose=True, show_plot=False)
pdf_kde = Z_kde / Z_kde.sum()
plt.tight_layout()
plt.show()

## 2. SGLD and EBM training setup

Stochastic Gradient Langevin Dynamics (SGLD) is used to sample from the energy model. The EBM is trained by contrastive divergence: minimize energy on data and maximize energy on generated samples.

In [ ]:
@tf.function
def sgld_sample(E, x_initial, num_steps, step_size, std_dev, clip_thresh=tf.constant(0.1)):
    """SGLD sampling: x_{k+1} = x_k - (step_size/2) * grad E(x_k) + noise."""
    x_k = x_initial
    for _ in range(num_steps):
        with tf.GradientTape() as g:
            g.watch(x_k)
            energy = tf.math.reduce_sum(E(x_k))
        dE_dx = tf.clip_by_norm(g.gradient(energy, x_k), clip_thresh)
        x_k = x_k - (step_size / 2) * dE_dx + tf.random.normal(x_k.shape, mean=0.0, stddev=std_dev)
    return x_k

In [ ]:
class EBM1:
    def __init__(self, energy_fn, replay_buffer_size=10000):
        self.energy = energy_fn
        self.replay_buffer = []
        self.replay_buffer_ptr = 0
        self.replay_buffer_max = replay_buffer_size

    def sample_sgld(self, x_init, batch_size, num_steps_markov=tf.constant(25), step_size=tf.constant(10.0),
                    std_dev=tf.constant(0.005), clip_thresh=tf.constant(0.01)):
        return sgld_sample(self.energy, x_init, num_steps_markov, step_size, std_dev, clip_thresh=clip_thresh)

    def sample_replay_buffer(self, batch_size, uniform_bounds_lower, uniform_bounds_upper,
                             num_steps_markov=tf.constant(25), step_size=tf.constant(10.0),
                             std_dev=tf.constant(0.005), clip_thresh=tf.constant(0.01)):
        initial_points = []
        for _ in range(batch_size):
            if len(self.replay_buffer) < batch_size or np.random.uniform() >= 0.95:
                x_k = np.random.uniform(uniform_bounds_lower, uniform_bounds_upper, (1, 2))
            else:
                idx = np.random.choice(len(self.replay_buffer))
                x_k = tf.reshape(self.replay_buffer[idx], (1, self.replay_buffer[idx].shape[-1]))
            x_k = tf.convert_to_tensor(x_k, dtype=tf.float32)
            initial_points.append(x_k)
        initial_points = tf.concat(initial_points, 0)
        return sgld_sample(self.energy, initial_points, num_steps_markov, step_size, std_dev, clip_thresh=clip_thresh)

    def _insert_into_replay_buffer(self, data, batch_size):
        for j in range(batch_size):
            entry = tf.reshape(data[j], (1, data[j].shape[-1]))
            if len(self.replay_buffer) < self.replay_buffer_max:
                self.replay_buffer.append(entry)
            else:
                self.replay_buffer[self.replay_buffer_ptr] = entry
            self.replay_buffer_ptr = (self.replay_buffer_ptr + 1) % self.replay_buffer_max

    def fit(self, data, batch_size, num_epochs, optimizer, uniform_bounds_lower, uniform_bounds_upper,
            alpha=tf.constant(0.1), num_steps_markov=tf.constant(25), step_size=tf.constant(10.0),
            std_dev=tf.constant(0.005), clip_thresh=tf.constant(0.01), callbacks_energy=[]):
        n_train = data.shape[0]
        n_batches = n_train // batch_size
        dataset = tf.data.Dataset.from_tensor_slices(data).shuffle(n_train).batch(batch_size, drop_remainder=True).repeat()
        it = iter(dataset)
        for epoch in range(num_epochs):
            for i in range(n_batches):
                sample_data = next(it)
                sample_energy = self.sample_replay_buffer(batch_size, uniform_bounds_lower, uniform_bounds_upper,
                    num_steps_markov=num_steps_markov, step_size=step_size, std_dev=std_dev, clip_thresh=clip_thresh)
                self._insert_into_replay_buffer(sample_energy, batch_size)
                with tf.GradientTape() as g:
                    e_data = tf.math.reduce_mean(self.energy(sample_data, training=True))
                    e_samples = tf.math.reduce_mean(self.energy(sample_energy, training=True))
                    e_l2 = tf.math.reduce_mean(tf.square(self.energy(sample_data, training=True))) + tf.math.reduce_mean(tf.square(self.energy(sample_energy, training=True)))
                    loss = e_data - e_samples + alpha * e_l2
                grads = g.gradient(loss, self.energy.trainable_variables)
                optimizer.apply_gradients(zip(grads, self.energy.trainable_variables))
            print("Epoch {:2d}  CD: {:12.10f}".format(epoch, (e_data - e_samples).numpy()), end="  ")
            for name, fn in callbacks_energy:
                print("{}: {:12.10f}".format(name, fn(self.energy)), end="  ")
            print()

## 3. Build and train the EBM

Define a small MLP as the energy function (input dimension 2 for the plane) and train it on the two-moons data.

In [ ]:
# Bounds for moons (approximate data range)
bounds_lower = [-2.0, -1.5]
bounds_upper = [3.0, 2.0]

# EBM: 2D input -> scalar energy
input_ = keras.layers.Input(shape=(2,))
h1 = keras.layers.Dense(256, activation="relu")(input_)
h2 = keras.layers.Dense(256, activation="relu")(h1)
output = keras.layers.Dense(1)(h2)
energy_model = keras.Model(inputs=[input_], outputs=[output])
energy_model.summary()

In [ ]:
# Train EBM
ebm = EBM1(energy_model)
optimizer = keras.optimizers.Adam(learning_rate=1e-4)
BATCH_SIZE = 128
EPOCHS = 30
ebm.fit(X_moons.astype(np.float32), BATCH_SIZE, EPOCHS, optimizer, bounds_lower, bounds_upper,
        num_steps_markov=tf.constant(15), step_size=tf.constant(0.2), std_dev=0.1, clip_thresh=tf.constant(0.1))

## 4. Visualize the learned energy function

Plot the energy E(x) over the plane (low energy = high probability under the model).

In [ ]:
def plot_energy_contour(E, x_lim=(-2, 3), y_lim=(-1.5, 2), num_points=250):
    x = np.linspace(x_lim[0], x_lim[1], num_points)
    y = np.linspace(y_lim[0], y_lim[1], num_points)
    X, Y = np.meshgrid(x, y)
    grid = np.vstack((X.flatten(), Y.flatten())).T
    grid_t = tf.convert_to_tensor(grid, dtype=tf.float32)
    Z = E(grid_t).numpy().reshape(X.shape)
    plt.figure(figsize=(6, 5))
    plt.contourf(X, Y, Z, levels=20)
    plt.colorbar(label="Energy")
    plt.title("Energy function of the trained EBM")
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.tight_layout()
    plt.show()

plot_energy_contour(energy_model)

## 5. Generate and visualize samples

Use SGLD to sample from the trained EBM and plot the generated points.

In [ ]:
# Initial points from uniform distribution
n_gen = 2000
x_init = tf.convert_to_tensor(
    np.random.uniform(bounds_lower, bounds_upper, (n_gen, 2)).astype(np.float32)
)
samples = sgld_sample(energy_model, x_init, tf.constant(50), 0.2, 0.15, clip_thresh=tf.constant(0.1))
samples_np = samples.numpy()

plt.figure(figsize=(6, 5))
plt.scatter(samples_np[:, 0], samples_np[:, 1], alpha=0.5, s=10)
plt.title("Generated samples from the trained EBM (SGLD)")
plt.xlabel("x1")
plt.ylabel("x2")
plt.xlim(bounds_lower[0], bounds_upper[0])
plt.ylim(bounds_lower[1], bounds_upper[1])
plt.tight_layout()
plt.show()

## 6. Visualize the density defined by the energy

The EBM defines a density q(x) = exp(-E(x)) / Z. We approximate it on a grid and plot it.

In [ ]:
def plot_density_energy(E, x_lim=(-2, 3), y_lim=(-1.5, 2), num_points=250):
    x = np.linspace(x_lim[0], x_lim[1], num_points)
    y = np.linspace(y_lim[0], y_lim[1], num_points)
    X, Y = np.meshgrid(x, y)
    grid = np.vstack((X.flatten(), Y.flatten())).T
    grid_t = tf.convert_to_tensor(grid, dtype=tf.float32)
    Z = np.exp(-E(grid_t).numpy().reshape(X.shape))
    Z = Z / Z.sum()
    plt.figure(figsize=(6, 5))
    plt.contourf(X, Y, Z, levels=50)
    plt.colorbar(label="Density")
    plt.title("Density q(x) = exp(-E(x)) / Z from the trained EBM")
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.tight_layout()
    plt.show()
    return Z

pdf_energy = plot_density_energy(energy_model)

## 7. Evaluate generative quality: KL divergence and Jensen-Shannon distance

Compute the KL divergence (using `tf.keras.losses.KLDivergence`) and the Jensen-Shannon distance (using `scipy.spatial.distance.jensenshannon`) between the data density (KDE) and the model density (from the energy function) on the same grid.

In [ ]:
# Use same grid for both densities
x_lim = (-2, 3)
y_lim = (-1.5, 2)
num_pts = 250
x_s = np.linspace(x_lim[0], x_lim[1], num_pts)
y_s = np.linspace(y_lim[0], y_lim[1], num_pts)
XX, YY = np.meshgrid(x_s, y_s)
grid_flat = np.vstack((XX.flatten(), YY.flatten())).T

# Data density (KDE) on grid, normalized
Z_data = kernel(grid_flat.T)
pdf_data_grid = Z_data.reshape(XX.shape)
pdf_data_grid = pdf_data_grid / pdf_data_grid.sum()

# Model density on same grid
grid_t = tf.convert_to_tensor(grid_flat, dtype=tf.float32)
Z_model = np.exp(-energy_model(grid_t).numpy()).reshape(XX.shape)
pdf_model_grid = Z_model / Z_model.sum()

# Flatten for scalar metrics
p_flat = pdf_data_grid.flatten().astype(np.float32)
q_flat = pdf_model_grid.flatten().astype(np.float32)
# Add small epsilon to avoid log(0) in KL
eps = 1e-10
p_flat = np.clip(p_flat, eps, 1.0)
q_flat = np.clip(q_flat, eps, 1.0)
p_flat = p_flat / p_flat.sum()
q_flat = q_flat / q_flat.sum()

# KL divergence: KL(data || model) using tf.keras.losses.KLDivergence
kl_loss = keras.losses.KLDivergence()
kl_div = kl_loss(tf.constant(p_flat), tf.constant(q_flat)).numpy()
print("KL divergence (data || model): {:.6f}".format(kl_div))

# Jensen-Shannon distance
js_dist = scipy.spatial.distance.jensenshannon(p_flat, q_flat)
print("Jensen-Shannon distance: {:.6f}".format(js_dist))